# Library import

In [2]:
# For eigendecomposition
using LinearAlgebra
#Includes Poisson distribution
using Distributions
# Plots
using Plots
# Symbolic computation
using Symbolics
# Solvers
using Groebner
using NLsolve # non-linear solver for extinction probabilities
# Numerical integration
using QuadGK
# Differentiation
using ForwardDiff
# Saving simulation results
using CSV, DataFrames
# Optimisation
using Optim
# Measuring elapsed time of simulations
using Dates
# Sampling
using StatsBase
# Maths notation
using LaTeXStrings

## Constants

In [3]:
# Sample size
k = 2
# Mean offspring matrix
poisson_parameters = [4 2; 1 3]
# Last generation
max_t = 20
# Type of root ancestor
i0 = 1
generation0 = zeros(Int, k)
generation0[i0] = 1
# Error bars constant
t = quantile(TDist(999), 1 - 0.025)

1.9623414611334569

# Eigen decomposition

Eigendecomposition of the mean offspring matrix.

In [4]:
# Eigendecomposition
eigen_decom = eigen(poisson_parameters)
eig_value = eigen_decom.values[2]
right = eigen_decom.vectors[:, 2]
left = eigen(poisson_parameters').vectors[:,2]
# Normalising eigenvectors
sm_left = left[1]+left[2] 
left = left./sm_left
sm_both = right[1]*left[1]+right[2]*left[2]
right = right./sm_both

2-element Vector{Float64}:
 1.3333333333333333
 0.6666666666666666

## Extinction calculation

In [5]:
euler = exp(1)
# Add variables for the mean offspring matrix values for brevity of code
lambda11 = poisson_parameters[1,1]
lambda12 = poisson_parameters[1,2]
lambda21 = poisson_parameters[2,1]
lambda22 = poisson_parameters[2,2]

# Variables
@variables s y1 y2
# Functions
@variables phi1(s) phi2(s) 

# Generating functions for both types
gen_func1 = euler^(-lambda11*(1-phi1)-lambda12*(1-phi2))
gen_func2 = euler^(-lambda21*(1-phi1)-lambda22*(1-phi2))

# Creating the equations f(q) - q = 0
q1 = substitute(gen_func1, Dict(phi1 => y1, phi2 => y2)) - y1
q2 = substitute(gen_func2, Dict(phi1 => y1, phi2 => y2)) - y2

# Converting these symbolic equations into a function
q1_func = eval(Symbolics.build_function(q1, y1, y2))
q2_func = eval(Symbolics.build_function(q2, y1, y2))

#3 (generic function with 1 method)

In [6]:
# Define the nonlinear system
function system!(F, vars)
    y1, y2 = vars  
    F[1] = q1_func(y1, y2)  
    F[2] = q2_func(y1, y2)  
end

# Initial guess for [x, y]
initial_guess = [0.5, 0.5]

# Solve the system
result = nlsolve(system!, initial_guess)

# Extract the solution
qs = result.zero

2-element Vector{Float64}:
 0.002604153130937961
 0.019467820792591555

## Joint distribution based on the generating functions

In [7]:
# Number of offspring to truncate at
max_offspring = 10

function compute_dist(gener)
    prob_matrix = zeros(Float64, max_offspring, max_offspring)
#     Probability of having i offspring of type 1
    for i in 0:max_offspring-1
#         Probability of having j offspring of type 2
        for j in 0:max_offspring-1
            deriv = gener
            if i != 0
                deriv = (Differential(y1)^i)(deriv)
            end
            if j != 0
                deriv = (Differential(y2)^j)(deriv)
            end
            deriv_full = expand_derivatives(deriv)
#             Substitute 0 for both arguments of the generating funciton after differentiating
            deriv_val = substitute(deriv_full, Dict(y1 => 0,y2 => 0))
            prob_matrix[i+1, j+1] = Symbolics.symbolic_to_float(deriv_val) / factorial(i) / factorial(j)
        end
    end
    return prob_matrix
end

compute_dist (generic function with 1 method)

### Generating functions of the Harris-Sevastyanov transformation

In [8]:
function gen_Func1(s1, s2)
    return (euler^(-lambda11*(1-s1)*(1-qs[1])-lambda12*(1-s2)*(1-qs[2]))-qs[1])/(1-qs[1])
end

function gen_Func2(s1, s2)
    return (euler^(-lambda21*(1-s1)*(1-qs[1])-lambda22*(1-s2)*(1-qs[2]))-qs[2])/(1-qs[2])
end

gen_Func2 (generic function with 1 method)

In [9]:
prob_matrix = []
all_probs = []
# Both generating functions for the Harris-Sevastyanov transformation
generatings = [gen_Func1, gen_Func2]
for i in 1:2
    current_matrix = compute_dist(generatings[i](y1, y2))
    # Replace any negative entries (e.g. from rounding errors) with 0
    current_matrix .= max.(current_matrix, 0.0)
    push!(prob_matrix, current_matrix)
    # Flatten the matrix into a 1D vector
    current_probs = vec(current_matrix)
    # Normalize the probabilities to sum to 1
    probs_normalised = current_probs / sum(current_probs)
    println("The matrix entries sum up to ", round(sum(current_probs), digits=2))
    push!(all_probs, probs_normalised)
end

The matrix entries sum up to 0.99
The matrix entries sum up to 1.0


In [15]:
function sample_Z(probabilities::Vector{Float64}, n_samples::Int)
#     Sample based on the supplied probabilities
    sampled = sample(1:length(probabilities), Weights(probabilities), n_samples)
#     Change to 0-based indexing
    sampled .-= 1
#     Compute which number of offspring of each type the probability corresponds to
    rows = sampled .÷ max_offspring   
    cols = sampled .% max_offspring  
    output = vcat(rows', cols')  
    return output
end

sample_Z (generic function with 1 method)

# Expected value of the maximum for HS transform

Below, we will approxiamte
$$\mathbb{E}(\max_{i \in \{1,2\}}(\frac{1}{|\boldsymbol{Y}_1^1|}, \frac{1}{|\boldsymbol{Y}_1^2|})); \qquad\mathbb{E}(\max_{i \in \{1,2\}}(\frac{1}{|\boldsymbol{Y}_1^1|^2}, \frac{1}{|\boldsymbol{Y}_1^2|^2})); 
\qquad \mathbb{E}(\max_{i \in \{1,2\}} (|\boldsymbol{Y}_1^1|, |\boldsymbol{Y}_1^2|))$$

In [17]:
# Number of samples to estimate the quntities above
max_estim = 1000
# Samples of a process started from a particle of type 1
sampled1 = sum(sample_Z(all_probs[1], max_estim), dims=1)
# Samples from a process started from a particle of type 2
sampled2 = sum(sample_Z(all_probs[2], max_estim), dims=1)

# Maximum and minimum over types
supY = max.(sampled1, sampled2)
infY = min.(sampled1, sampled2)
# approximations for the quantities above
e_max = mean(supY)
harmonic_mean1 = mean(ones(max_estim)./vec(infY))
harmonic_mean2 = mean(ones(max_estim)./(vec(infY)).^k)

1×1000 Matrix{Int64}:
 12  3  7  8  4  2  5  5  3  6  3  5  …  5  5  4  5  3  3  8  3  4  1  3  3

## First 4 moments of W approximation

For $\phi(s) = \mathbb{E}(e^{-sW})$, $f$ a generating function and $\lambda$ the maximal eigenvalue of the mean offspring matrix, we have that $\phi(\lambda s) = f(\phi(s))$.

In [22]:
# Derivative of the \phi(\lambda s)
function lhs_derivative(equation, n)
    return eig_value^n*(Differential(s)^n)(equation)
end

# Differentiate both sides of the equation above, evaluate at 0 and solve for moments of W
function differentiate(lhs, rhs, n)
    der_l = lhs_derivative(lhs, n)
    der_r = expand_derivatives((Differential(s)^n)(rhs))
    der = der_r - der_l
#     Replace the derivative with a dummy variable to be solved for
    der = substitute(der, (Differential(s)^n)(phi1) => y1)
    der = substitute(der, (Differential(s)^n)(phi2) => y2)
#     Substitute all lower derivatives with values that are already know in the recursion
    for less in (n-1):-1:1
        der = substitute(der, (Differential(s)^less)(phi1) => estim1[less])
        der = substitute(der, (Differential(s)^less)(phi2) => estim2[less])
    end
#     The value \phi at 0 is replaced with 1
    der = substitute(der, Dict(phi1 => 1, phi2 => 1))
    return der
end

lhs_derivative (generic function with 1 method)

In [24]:
# Calculating up to 4th moment
# First moment
estim1 = fill(right[1], 4)
estim2 = fill(right[2], 4)
for deriv in 2:4
    eq1 = differentiate(phi1, gen_func1, deriv)
    eq2 = differentiate(phi2, gen_func2, deriv)
    moments = symbolic_linear_solve([eq1 ~ 0, eq2 ~ 0], [y1, y2])
    estim1[deriv] = moments[1]
    estim2[deriv] = moments[2]
end

# Expectations and variances
E_w1_square = estim1[2]
E_w2_square = estim2[2]
var_w1 = E_w1_square - right[1]^2
var_w2 = E_w2_square - right[2]^2
var_w1_square = estim1[4]-estim1[2]^2
var_w2_square = estim2[4]-estim2[2]^2

# Calculation of variance of W from theory

The full formula can be found in "The Theory of Brancing Processes" by Theodore E. Harris (1964) in Chapter 2.9.

In [26]:
C0 = [1 0;0 0]
# Variance of first generation
V1 = Diagonal(poisson_parameters[1,1:2])
V2 = Diagonal(poisson_parameters[2,1:2])
var_matrix = zeros(2, max_t)

M1 = right*left'
# Variance for starting type 1
gen0 = [1;0]
delta = gen0'*inv(I(2)-poisson_parameters*eig_value^(-2))
varW1_before = eig_value^(-2)*M1'*(delta[1]*V1+delta[2]*V2)*M1
# Variance for starting type 2
gen0 = [0;1]
delta = gen0'*inv(I(2)-poisson_parameters*eig_value^(-2))
varW2_before = eig_value^(-2)*M1'*(delta[1]*V1+delta[2]*V2)*M1

matrix_of_nu = [left[1]^2 left[1]*left[2]; left[1]*left[2] left[2]^2]

# Rescaling variances due to Harris' different definitions
varW1 = varW1_before ./ matrix_of_nu
varW2 = varW2_before ./ matrix_of_nu
w1_square = varW1[1,1]+right[1]^2
w2_square = varW2[2,2]+right[2]^2

2×20 Matrix{Float64}:
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0

# Upper bound

In [30]:
# Number of generations of interest
max_t = 10
# Maximum epsilon for upper bound
max_epsilon = min(right[1], right[2])
print("Epsilon cannot be larger than: ", max_epsilon)

10

`bound_constant` calculates the constant from the upper bound, i.e.
$$
(1-\max_{i \in [2]}q_i)^{-2} \left(
\frac{\max_{i \in [2]} E((W^{(i)})^2)+ \epsilon}{(\min_{i \in [2]} E(W^{(i)})- \epsilon)^2} 
+ \sum_{j \in [2]}\frac{\max_{i \in [2]} Var((W^{(i)})^j)}{\epsilon^2 (1-\max_{i \in [2]} q_i)}\right)
$$

where $[c] := \{1,2,...,c\}$.  

In [32]:
function bound_constant(ep)
    good_term = (max(w1_square, w2_square)+ep)/(min(right[1], right[2]) - ep)^k
    bad_term = (max(var_w1_square, var_w2_square)+max(varW1[1,1], varW2[2,2]))/(ep^2*(1-max(qs[1], qs[2])))
    for_both = (1-max(qs[1], qs[2]))^(-2)
    constant = for_both*(good_term+bad_term)
end

# Find optimal epsilon in the range of its possible values
result = optimize(bound_constant, 0.1*max_epsilon, 0.99*max_epsilon)
optimal_eps = Optim.minimizer(result)

# Multiply the optimal constant by the harmonic moment to obtain the bound
constant = bound_constant(optimal_eps)
bound = zeros(max_t)
for i in 1:max_t
    bound[i] =   harmonic_mean1^i*constant
end

# Replacing anything greater than 1 with 1
bound[bound .> 1] .= 1

# Lower bound

In [34]:
lower_epsilon = min(w1_square, w2_square)
print("For the lower bound, epsilon cannot be larger than: ", lower_epsilon)

For the lower bound, epsilon cannot be larger than: 0.6038647342995169

Computing the two constants needed for the lower bound:
$$
    (\frac{\min_{i \in [2]} E((W^{(i)})^2)- \epsilon}{(\max_{i \in [2]} E(W^{(i)})+ \epsilon)^2} )
    (1-\max_{i \in [2]} q_i)^2; \qquad 
    \frac{\min_{i \in [2]} E((W^{(i)})^2)- \epsilon}{(\max_{i \in [2]} E(W^{(i)})+ \epsilon)^2} 
    \sum_{j \in [2]}\frac{\max_{i \in [2]} Var((W^{(i)})^j)}{\epsilon^2 (1-\max_{i \in [2]} q_i)}
    (1-\max_{i \in [2]} q_i)^{-2}.
$$

In [35]:
# This function returns either of the constants above
function lower_bound(ep; minus = false, plus = false)
    max_extinct = max(qs[1], qs[2])
    C2 = (min(w1_square, w2_square) - ep)/(max(right[1], right[2])+ep)^2
    constant_lower = (1-max_extinct)^2*C2
    C3 = (max(var_w1_square, var_w2_square)+max(varW1[1,1], varW2[2,2]))/(ep^2*(1-max_extinct))
    extra_constant = C3*constant_lower
    if minus
        res = constant_lower
    end
    if plus
        res = extra_constant
    end
    return res
end


# Function that uses the constants and harmonic means to compute the lower bound
function actual_bound(ep)
    positive_part = e_max^(-current_time)*lower_bound(ep, minus = true)
    negative_part = harmonic_mean2^(current_time)*lower_bound(ep, plus = true)
    return -positive_part + negative_part
end

lower_bound (generic function with 1 method)

In [37]:
# Optimising epsilon for each time t
current_time = 1
optimal_epsilons = zeros(max_t)
for times in 1:max_t
    result0 = optimize(actual_bound, 0.1*lower_epsilon, 0.99*lower_epsilon)
    optimal_epsilons[times] = Optim.minimizer(result0)
    current_time += 1
end

In [40]:
# Calculating the lower bound using optimal epsilons
bound0 = zeros(max_t)
for i in 1:max_t
    positive_part = e_max^(-i)*lower_bound(optimal_epsilons[i],plus = true)
    negative_part = harmonic_mean2^i*lower_bound(optimal_epsilons[i], minus = true)
    bound0[i] = positive_part - negative_part 
end

# Probability of coalescence by simulation

In [46]:
# The number of extra generations the process is evolving for after 
extra_gens = 6
# Number of simulations
nb_sims = 1000
# Number of generations of interest for the probability distribution
max_t_sim = 6
# Empty vector for the proportion of the samples that coalesced
proportion_coalesced = ones(max_t_sim)
error_bar = zeros(max_t_sim)

6-element Vector{Float64}:
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0

In [47]:
for gen_ancestry in 2:max_t_sim
#     If they share an ancestor
    same_ancestor = fill(-1, nb_sims)
    which_alive = ones(Int, nb_sims)
    for sim in 1:nb_sims
#         Population sizes for the future
        pop_sizes = zeros(Int, 2, max_t_sim+extra_gens)
        pop_sizes[i0,1] = 1
        for gen in 1:(gen_ancestry-1)
            for typ in 1:2
                mean_now = poisson_parameters[typ,1:2]
                pop_now = pop_sizes[typ, gen]
                if sum(pop_now) != 0
            #         Children type 1
                    sample1 = rand(Poisson(mean_now[1]), pop_now)
            #         Children type 2
                    sample2 = rand(Poisson(mean_now[2]), pop_now)
                    pop_sizes[1, gen+1] += sum(sample1)
                    pop_sizes[2, gen+1] += sum(sample2)
                end
            end
        end
#         Recording if a population is extinct
        if sum(pop_sizes[1:2, gen_ancestry]) == 0
            which_alive[sim] = 0
#         Do not conitnue if the population is extinct
        else sum(pop_sizes[1:2, gen_ancestry]) != 0
                # Enumerate all the possible ancestors at the starting generation
            ancestors1 = collect(1:(pop_sizes[1, gen_ancestry]))
            ancestors2 =collect((pop_sizes[1, gen_ancestry]+1):(pop_sizes[1, gen_ancestry]+pop_sizes[2, gen_ancestry]))
    #         For all generations in the future of the starting generation
            for gen in (gen_ancestry):(max_t_sim+extra_gens-1)
                ancestors = []
                if sum(pop_sizes[1:2, gen]) != 0
                    for typ in 1:2
                        if typ == 1
                            current_ancestors = ancestors1 
                        else
                            current_ancestors = ancestors2
                        end
                        mean_now = poisson_parameters[typ,1:2]
                        pop_now = pop_sizes[typ, gen]
                #         Children type 1
                        sample1 = rand(Poisson(mean_now[1]), pop_now)
                #         Children type 2
                        sample2 = rand(Poisson(mean_now[2]), pop_now)
                        pop_sizes[1, gen+1] += sum(sample1)
                        pop_sizes[2, gen+1] += sum(sample2)
           #         Ancestors in the new generation
                        new_ancestors1 = vcat(fill.(current_ancestors, sample1)...)
                        new_ancestors2 = vcat(fill.(current_ancestors, sample2)...)
                        push!(ancestors, new_ancestors1)
                        push!(ancestors, new_ancestors2)
                    end
                    #             Ancestors of all type 1 and 2
                    ancestors1 = vcat(ancestors[1], ancestors[3])
                    ancestors2 = vcat(ancestors[2], ancestors[4])
                end
            end
#             Recording which simulations died out
            if sum(pop_sizes[1:2,max_t_sim+extra_gens]) == 0
                which_alive[sim] = 0
            else
                all_ancestors = vcat(ancestors1, ancestors2)
        #         sample a pair from the last generation
                 pair=rand(1:length(all_ancestors),2) 
                same_ancestor[sim] = (all_ancestors[pair[1]] == all_ancestors[pair[2]])
            end 
        end
    end
#     Removing extinct simulations
    without_dead = same_ancestor[which_alive .== 1]
#     Calculating the proportion that coalesced
    proportion_coalesced[gen_ancestry] = mean(without_dead)
    error_bar[gen_ancestry] = t*std(without_dead)/sqrt(length(without_dead))
end

2025-10-31T18:08:00.145

#### Saving and loading simulation data

In [48]:
# CSV.write("data/poisson_1234_1000sims_6extra_len6.csv", DataFrame(Vector=proportion_coalesced))
# CSV.write("data/poisson_1234_error_1000sims_6extra_len6.csv", DataFrame(Vector=error_bar))

# proportion_coalesced = CSV.read("data/prop_coalesced1000_10gens_8extra.csv", DataFrame; header=false)[:,1]
# error_bar = CSV.read("data/error_genealogy1000_10gens_8extra.csv", DataFrame; header = false)[:,1]

"poisson_1234_error_1000sims_6extra_len6.csv"

### Loading the estimate for coalescence time distribution based on the inverse fourier transform

In [54]:
# formula = CSV.read("data/formula_1234_1000sim.csv", DataFrame, header=false)[:,1]
# err_formula = CSV.read("data/formula_err_1234_1000sims.csv", DataFrame, header=false)[:,1]

10-element Vector{Float64}:
 0.0
 0.009996938675861475
 0.0037588749747982375
 0.0010011301956207154
 0.00025919224082279806
 4.820050380635761e-5
 1.0053294094338772e-5
 2.0580382547353844e-6
 4.0449897913902257e-7
 8.099153947135731e-8

## Plotting the bounds and simulation results

In [57]:
plot(1:length(proportion_coalesced), ones(length(proportion_coalesced)).- proportion_coalesced, 
    yerror = error_bar, label = "Simulation", markerstrokecolor = :lightblue, color=:blue)
plot!(1:max_t, ones(length(bound)) .- bound, 
#     title = "Bounds for coalescence probability", 
    xlabel = "Generation", 
    ylabel = L"\mathbb{P}(X_{T,2} < t\mid\mathbf{Z}_{T} \geq 2)", 
    label = "Lower bound", linestyle = :dash)
plot!(1:length(formula), 1 .- formula, label = "Formula", yerror = err_formula,  legend=:bottomright,#legend=:inside, 
    markerstrokecolor = :lightgreen, color=:green, tickfontsize=10, guidefontsize=14)
p = plot!(1:max_t, ones(length(bound0)).- bound0, label = "Upper bound", linestyle=:dash, legendfont=14)
# savefig(p, "images/bounds_poisson_1234.pdf") 

"/home/janique/Documents/w_density/images/bounds_poisson_1234.pdf"